# Multi-Agent Deep Research for Cat Nutrition and Obesity

Deep research is not one enormous prompt. It is a composition of familiar patterns:

```text
scope -> delegate -> search -> verify -> write -> audit -> evaluate
```

In this notebook, every reasoning actor is built with LangChain's `create_agent`.
A small LangGraph workflow controls the lifecycle, parallel-friendly delegation,
optional human clarification, persistence, and deterministic checks.

The worked example asks:

> What does current evidence say about preventing and safely managing obesity in adult indoor cats?

This is an educational research workflow. It does not diagnose an animal or create
an individualized feeding or weight-loss plan.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain when multiple agents improve context isolation and role specialization.
- Build named `create_agent` workers and expose them as high-level tools.
- Use structured outputs as contracts between research stages.
- Combine agentic loops with a deterministic LangGraph workflow.
- Pause and resume a workflow with `interrupt()` and `Command(resume=...)`.
- Track source URLs through research, verification, writing, and citation auditing.
- Reason about research quality, latency, and API cost as explicit design tradeoffs.


## Table of Contents

- **Breakout Room #1: Specialized Agents and Delegation**
  - Task 1: Environment Setup
  - Task 2: Define Typed Handoff Contracts
  - Task 3: Configure Tavily Search and Extract
  - Task 4: Build Specialized Research Workers
  - Task 5: Wrap Workers as Supervisor Tools
  - Task 6: Build the Research Supervisor
  - Activity #1: Add a New Specialist
- **Breakout Room #2: End-to-End Deep Research Workflow**
  - Task 7: Build the Scoper and Clarification Path
  - Task 8: Build Verification, Writing, and Evaluation Agents
  - Task 9: Define LangGraph State and Nodes
  - Task 10: Compile, Visualize, and Stream the Workflow
  - Task 11: Audit Citations Deterministically
  - Task 12: Run the Full Deep Research System
  - Activity #2: Compare Research Depth and Cost
  - Advanced Build: Add a Local-Corpus Specialist


---
# Breakout Room #1
## Specialized Agents and Delegation

A multi-agent system is useful when independent actors need different prompts,
tools, context, or optimization targets.

Our supervisor does not search the web directly. It delegates bounded research
tasks to workers and receives compact, typed findings. This keeps low-level search
traces out of the supervisor's context.


## Task 1: Environment Setup

From the `04_Multi_Agent_Systems` folder:

```bash
uv sync
```

Then select the environment created by uv as this notebook's kernel.

Required accounts:

- OpenAI API
- Tavily API

LangSmith tracing is optional but strongly recommended for inspecting nested agents.


### Imports


In [ ]:
from __future__ import annotations

import json
import os
import re
from datetime import date
from getpass import getpass
from typing import Literal, TypedDict
from urllib.parse import urlsplit, urlunsplit

from IPython.display import Image, Markdown, display
from pydantic import BaseModel, Field

from langchain.agents import create_agent
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
)
from langchain.messages import ToolMessage
from langchain.tools import tool
from langchain_core.runnables import RunnableConfig
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilyExtract, TavilySearch

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


### API Keys, Models, and Research Budgets

The environment variables make the notebook easy to tune without editing code.
Research depth is an application policy, not a mysterious property of the model.

`AIM_SEARCH_CALL_LIMIT` limits Tavily searches per worker invocation.
`AIM_WORKER_MODEL_CALL_LIMIT` bounds each worker's ReAct loop.


In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Tavily API Key: ")

os.environ.setdefault(
    "LANGSMITH_PROJECT",
    "aim-session-4-multi-agent-deep-research",
)

TODAY = date.today().isoformat()
CHAT_MODEL_NAME = os.environ.get("AIM_CHAT_MODEL", "gpt-5.4-mini")
WRITER_MODEL_NAME = os.environ.get("AIM_WRITER_MODEL", CHAT_MODEL_NAME)
SEARCH_DEPTH = os.environ.get("AIM_SEARCH_DEPTH", "advanced")
SEARCH_CALL_LIMIT = int(os.environ.get("AIM_SEARCH_CALL_LIMIT", "3"))
EXTRACT_CALL_LIMIT = int(os.environ.get("AIM_EXTRACT_CALL_LIMIT", "2"))
WORKER_MODEL_CALL_LIMIT = int(
    os.environ.get("AIM_WORKER_MODEL_CALL_LIMIT", "8")
)

llm = ChatOpenAI(model=CHAT_MODEL_NAME)
writer_llm = ChatOpenAI(model=WRITER_MODEL_NAME)

print(f"Date: {TODAY}")
print(f"General model: {CHAT_MODEL_NAME}")
print(f"Writer model: {WRITER_MODEL_NAME}")
print(f"Search depth: {SEARCH_DEPTH}")
print(f"Search calls per worker: {SEARCH_CALL_LIMIT}")
print(f"LangSmith tracing: {os.environ.get('LANGSMITH_TRACING', 'false')}")


## Task 2: Define Typed Handoff Contracts

Free-form prose is difficult to validate and pass between application stages.
Pydantic models make the handoff contracts explicit.

Notice that source URLs are first-class data. The report writer will not be allowed
to invent a new source that did not survive verification.


In [ ]:
SpecialistName = Literal["guidelines", "evidence"]
ConfidenceLevel = Literal["high", "medium", "low"]


class ResearchTask(BaseModel):
    task_id: str = Field(description="Stable short identifier such as task-1")
    question: str = Field(description="Focused research question for one worker")
    specialist: SpecialistName
    rationale: str


class ResearchBrief(BaseModel):
    title: str
    objective: str
    audience: str
    scope_in: list[str]
    scope_out: list[str]
    safety_constraints: list[str]
    tasks: list[ResearchTask] = Field(min_length=3, max_length=3)


class ScopeDecision(BaseModel):
    needs_clarification: bool
    clarification_question: str | None = None
    assumptions: list[str] = Field(default_factory=list)
    brief: ResearchBrief | None = None


class SourceRecord(BaseModel):
    url: str
    title: str
    publisher: str
    published_date: str | None = None
    source_type: Literal[
        "guideline",
        "peer_reviewed",
        "government",
        "university",
        "professional_org",
        "other",
    ]
    authority_reason: str
    relevant_excerpt: str
    accessed_on: str


class ClaimRecord(BaseModel):
    claim: str
    source_urls: list[str] = Field(min_length=1)
    confidence: ConfidenceLevel
    caveat: str


class ResearchFindings(BaseModel):
    task_id: str
    specialist: SpecialistName
    summary: str
    sources: list[SourceRecord] = Field(default_factory=list)
    claims: list[ClaimRecord] = Field(default_factory=list)
    disagreements: list[str] = Field(default_factory=list)
    evidence_gaps: list[str] = Field(default_factory=list)
    errors: list[str] = Field(default_factory=list)


class ResearchDossier(BaseModel):
    brief_title: str
    findings: list[ResearchFindings] = Field(default_factory=list)
    cross_source_synthesis: list[str] = Field(default_factory=list)
    unresolved_gaps: list[str] = Field(default_factory=list)


class RejectedSource(BaseModel):
    url: str
    reason: str


class VerificationReport(BaseModel):
    approved_sources: list[SourceRecord] = Field(default_factory=list)
    rejected_sources: list[RejectedSource] = Field(default_factory=list)
    verified_claims: list[ClaimRecord] = Field(default_factory=list)
    unsupported_claims: list[str] = Field(default_factory=list)
    disagreements: list[str] = Field(default_factory=list)
    evidence_gaps: list[str] = Field(default_factory=list)
    verification_summary: str


class FinalReport(BaseModel):
    title: str
    executive_summary: str
    report_markdown: str
    limitations: list[str]
    medical_safety_note: str
    citations: list[str] = Field(
        description="Ordered list of source URLs cited in report_markdown"
    )


class CitationAudit(BaseModel):
    cited_urls: list[str]
    unknown_urls: list[str]
    duplicate_urls: list[str]
    approved_but_uncited: list[str]
    missing_marker_numbers: list[int]
    passed: bool


class ReportEvaluation(BaseModel):
    coverage: int = Field(ge=1, le=5)
    synthesis: int = Field(ge=1, le=5)
    source_quality: int = Field(ge=1, le=5)
    citation_integrity: int = Field(ge=1, le=5)
    uncertainty_handling: int = Field(ge=1, le=5)
    medical_safety: int = Field(ge=1, le=5)
    strengths: list[str]
    improvements: list[str]
    passed: bool


#### ❓Question #1

Why is a typed handoff safer than asking each agent to return "a useful summary"?
Which fields are application guarantees, and which still depend on model judgment?

##### ✅ Answer:

_Write your answer here._


## Task 3: Configure Tavily Search and Extract

Search discovers candidate sources. Extract reads selected pages more deeply.
We deliberately do not use Tavily's Research API because the goal is to build the
multi-agent research behavior ourselves.


In [ ]:
tavily_search = TavilySearch(
    max_results=5,
    topic="general",
    search_depth=SEARCH_DEPTH,
    include_answer=False,
    include_raw_content=False,
)

tavily_extract = TavilyExtract(
    extract_depth="advanced",
    format="markdown",
    chunks_per_source=3,
)

print(tavily_search.name)
print(tavily_extract.name)


The search and extraction tools return URLs in tool messages. We will retain only
source URLs that were actually observed in those tool results. This does not prove
a claim is correct, but it prevents an agent from adding an entirely unobserved URL
to the source ledger.


In [ ]:
URL_PATTERN = re.compile(r"https?://[^\s\]\[()<>{}\"']+")


def normalize_url(url: str) -> str:
    '''Normalize a URL for deterministic ledger comparison.'''
    cleaned = url.strip().rstrip(".,;:!?)")
    parts = urlsplit(cleaned)
    path = parts.path.rstrip("/") or "/"
    return urlunsplit(
        (
            parts.scheme.lower(),
            parts.netloc.lower(),
            path,
            parts.query,
            "",
        )
    )


def urls_in_value(value: object) -> set[str]:
    '''Extract normalized URLs from strings or JSON-like tool content.'''
    text = value if isinstance(value, str) else json.dumps(value, default=str)
    return {normalize_url(match) for match in URL_PATTERN.findall(text)}


def observed_tool_urls(agent_result: dict) -> set[str]:
    observed: set[str] = set()
    for message in agent_result.get("messages", []):
        if isinstance(message, ToolMessage):
            observed.update(urls_in_value(message.content))
    return observed


def sanitize_findings(
    findings: ResearchFindings,
    observed_urls: set[str],
) -> ResearchFindings:
    '''Drop sources and claims that cannot be traced to worker tool output.'''
    approved_sources = [
        source
        for source in findings.sources
        if normalize_url(source.url) in observed_urls
    ]
    approved_urls = {normalize_url(source.url) for source in approved_sources}

    approved_claims = []
    dropped_claims = []
    for claim in findings.claims:
        traced_urls = [
            url
            for url in claim.source_urls
            if normalize_url(url) in approved_urls
        ]
        if traced_urls:
            approved_claims.append(
                claim.model_copy(update={"source_urls": traced_urls})
            )
        else:
            dropped_claims.append(
                f"Untraced claim removed before supervision: {claim.claim}"
            )

    errors = list(findings.errors)
    if findings.sources and not approved_sources:
        errors.append(
            "The worker returned sources, but none of their URLs appeared "
            "in its search or extraction tool messages."
        )

    return findings.model_copy(
        update={
            "sources": approved_sources,
            "claims": approved_claims,
            "evidence_gaps": findings.evidence_gaps + dropped_claims,
            "errors": errors,
        }
    )


def sanitize_dossier(
    dossier: ResearchDossier,
    brief: ResearchBrief,
    observed_urls: set[str],
) -> ResearchDossier:
    '''Enforce worker URL provenance and record missing delegated tasks.'''
    task_by_id = {task.task_id: task for task in brief.tasks}
    sanitized_findings = []
    unexpected_findings = []
    for finding in dossier.findings:
        task = task_by_id.get(finding.task_id)
        if task is None:
            unexpected_findings.append(
                f"Unexpected findings returned for {finding.task_id}."
            )
            continue
        normalized_finding = finding.model_copy(
            update={"specialist": task.specialist}
        )
        sanitized_findings.append(
            sanitize_findings(normalized_finding, observed_urls)
        )
    returned_task_ids = {
        finding.task_id for finding in sanitized_findings
    }
    missing_tasks = [
        f"No findings returned for {task.task_id}: {task.question}"
        for task in brief.tasks
        if task.task_id not in returned_task_ids
    ]
    return dossier.model_copy(
        update={
            "brief_title": brief.title,
            "findings": sanitized_findings,
            "unresolved_gaps": (
                dossier.unresolved_gaps
                + unexpected_findings
                + missing_tasks
            ),
        }
    )


def sanitize_verification(
    verification: VerificationReport,
    observed_urls: set[str],
) -> VerificationReport:
    '''Keep only sources and claims traced to verifier tool output.'''
    approved_sources = [
        source
        for source in verification.approved_sources
        if normalize_url(source.url) in observed_urls
    ]
    approved_urls = {
        normalize_url(source.url) for source in approved_sources
    }
    verified_claims = []
    unsupported_claims = list(verification.unsupported_claims)
    for claim in verification.verified_claims:
        traced_urls = [
            url
            for url in claim.source_urls
            if normalize_url(url) in approved_urls
        ]
        if traced_urls:
            verified_claims.append(
                claim.model_copy(update={"source_urls": traced_urls})
            )
        else:
            unsupported_claims.append(
                f"Untraced verifier claim removed: {claim.claim}"
            )

    evidence_gaps = list(verification.evidence_gaps)
    if verification.approved_sources and not approved_sources:
        evidence_gaps.append(
            "The verifier approved sources, but none appeared in its tool output."
        )

    return verification.model_copy(
        update={
            "approved_sources": approved_sources,
            "verified_claims": verified_claims,
            "unsupported_claims": unsupported_claims,
            "evidence_gaps": evidence_gaps,
        }
    )


## Task 4: Build Specialized Research Workers

Both workers use the same model and tools, but they have different search priorities
and evaluation criteria.

- The **guideline researcher** prioritizes professional and public-health guidance.
- The **evidence researcher** prioritizes peer-reviewed studies and actively tests
  common nutrition claims.

The workers return `ResearchFindings`, not final prose for the user.


In [ ]:
def worker_middleware():
    '''Create fresh operational limits for one research agent.'''
    return [
        ModelCallLimitMiddleware(
            run_limit=WORKER_MODEL_CALL_LIMIT,
            exit_behavior="end",
        ),
        ToolCallLimitMiddleware(
            tool_name=tavily_search.name,
            run_limit=SEARCH_CALL_LIMIT,
            exit_behavior="continue",
        ),
        ToolCallLimitMiddleware(
            tool_name=tavily_extract.name,
            run_limit=EXTRACT_CALL_LIMIT,
            exit_behavior="continue",
        ),
    ]


GUIDELINE_RESEARCHER_PROMPT = f'''
You are the guideline specialist in a multi-agent veterinary research system.
Today's date is {TODAY}.

Research only the assigned task. Prioritize primary or authoritative sources:
AAFP, AAHA, WSAVA, AVMA, government agencies, veterinary schools, and recognized
veterinary nutrition organizations. Search before making factual claims. Extract
promising pages when snippets are not enough.

Return exactly the ResearchFindings schema. Every source URL and every claim URL
must come from a search or extraction tool result in this run. Explain why each
source is authoritative. Record disagreement and missing evidence. Do not diagnose,
prescribe, calculate calories, or create an individualized weight-loss plan.
Your final response is the only information the supervisor receives.
'''

EVIDENCE_RESEARCHER_PROMPT = f'''
You are the evidence specialist in a multi-agent veterinary research system.
Today's date is {TODAY}.

Research only the assigned task. Prioritize peer-reviewed studies, systematic
reviews, consensus statements, university sources, and primary publications.
Challenge broad claims about wet versus dry food, meal frequency, activity,
energy restriction, body-condition scoring, and safe rates of feline weight loss.
Distinguish association from causation and guidelines from controlled evidence.
Search before making factual claims and extract promising pages when needed.

Return exactly the ResearchFindings schema. Every source URL and every claim URL
must come from a search or extraction tool result in this run. Preserve uncertainty,
conflicting findings, study limitations, and unanswered questions. Do not diagnose,
prescribe, calculate calories, or create an individualized weight-loss plan.
Your final response is the only information the supervisor receives.
'''

guideline_researcher = create_agent(
    model=llm,
    tools=[tavily_search, tavily_extract],
    system_prompt=GUIDELINE_RESEARCHER_PROMPT,
    middleware=worker_middleware(),
    response_format=ResearchFindings,
    name="guideline_researcher",
)

evidence_researcher = create_agent(
    model=llm,
    tools=[tavily_search, tavily_extract],
    system_prompt=EVIDENCE_RESEARCHER_PROMPT,
    middleware=worker_middleware(),
    response_format=ResearchFindings,
    name="evidence_researcher",
)

print("Created named guideline and evidence research agents.")


## Task 5: Wrap Workers as Supervisor Tools

This is the central subagent pattern:

```text
supervisor -> high-level worker tool -> create_agent worker -> Tavily tools
```

The supervisor sees one compact JSON result per worker call. It does not receive the
worker's entire message history.

The wrappers also turn worker failures into explicit gaps so one failed research
stream does not erase successful work from another stream.


In [ ]:
def failed_findings(
    task: ResearchTask,
    specialist: SpecialistName,
    error: Exception,
) -> ResearchFindings:
    return ResearchFindings(
        task_id=task.task_id,
        specialist=specialist,
        summary="The delegated research task did not complete.",
        evidence_gaps=[task.question],
        errors=[f"{type(error).__name__}: {error}"],
    )


async def run_worker(
    agent,
    task_json: str,
    specialist: SpecialistName,
) -> ResearchFindings:
    try:
        task = ResearchTask.model_validate_json(task_json)
    except Exception as exc:
        return ResearchFindings(
            task_id="invalid-task",
            specialist=specialist,
            summary="The supervisor supplied an invalid research task.",
            evidence_gaps=["The delegated task could not be parsed."],
            errors=[f"{type(exc).__name__}: {exc}"],
        )

    try:
        result = await agent.ainvoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": (
                            "Complete this delegated research task and return "
                            f"the required structured findings:\n{task_json}"
                        ),
                    }
                ]
            }
        )
        findings = ResearchFindings.model_validate(
            result["structured_response"]
        )
        findings = findings.model_copy(
            update={
                "task_id": task.task_id,
                "specialist": specialist,
            }
        )
        return sanitize_findings(
            findings,
            observed_tool_urls(result),
        )
    except Exception as exc:
        return failed_findings(task, specialist, exc)


@tool(
    "research_guidelines",
    description=(
        "Delegate one structured ResearchTask to the veterinary guideline "
        "specialist. Input must be the task's JSON representation."
    ),
)
async def research_guidelines(task_json: str) -> str:
    findings = await run_worker(
        guideline_researcher,
        task_json,
        "guidelines",
    )
    return findings.model_dump_json(indent=2)


@tool(
    "research_evidence",
    description=(
        "Delegate one structured ResearchTask to the peer-reviewed evidence "
        "specialist. Input must be the task's JSON representation."
    ),
)
async def research_evidence(task_json: str) -> str:
    findings = await run_worker(
        evidence_researcher,
        task_json,
        "evidence",
    )
    return findings.model_dump_json(indent=2)


#### ❓Question #2

What context remains isolated inside each worker? What information crosses the
worker-supervisor boundary? How does this affect cost and debuggability?

##### ✅ Answer:

_Write your answer here._


## Task 6: Build the Research Supervisor

The supervisor receives exactly three tasks from the scoper. It should emit all
independent worker calls in the same model turn when possible. LangChain's agent
tool node can execute multiple tool calls concurrently.

This remains agentic: the model chooses the worker for each task and may revise its
approach after reading worker outputs. Operational middleware prevents an unbounded
delegation loop.


In [ ]:
SUPERVISOR_PROMPT = f'''
You are the lead researcher coordinating a bounded multi-agent investigation.
Today's date is {TODAY}.

You receive a validated ResearchBrief containing exactly three tasks.
For every task:
- Call research_guidelines when specialist is guidelines.
- Call research_evidence when specialist is evidence.
- Pass the complete ResearchTask as JSON.

The tasks are independent. Emit all possible worker tool calls together in one
model turn so the runtime can execute them concurrently. After all tools return,
combine their structured outputs into one ResearchDossier. Preserve errors,
disagreements, and evidence gaps. Do not create new sources or URLs. Do not write
the final user report.
'''

research_supervisor = create_agent(
    model=llm,
    tools=[research_guidelines, research_evidence],
    system_prompt=SUPERVISOR_PROMPT,
    middleware=[
        ModelCallLimitMiddleware(run_limit=6, exit_behavior="end"),
        ToolCallLimitMiddleware(
            tool_name=research_guidelines.name,
            run_limit=3,
            exit_behavior="continue",
        ),
        ToolCallLimitMiddleware(
            tool_name=research_evidence.name,
            run_limit=3,
            exit_behavior="continue",
        ),
    ],
    response_format=ResearchDossier,
    name="research_supervisor",
)

print("Research supervisor ready.")


## 🏗️ Activity #1: Add a New Specialist

Add a third specialist without giving the supervisor more low-level tools.

Choose one:

- A source-methodology specialist that scores study design and sample quality
- A veterinary communication specialist that identifies misleading public claims
- A cost-and-feasibility specialist that compares practical interventions

Requirements:

1. Create a named `create_agent` worker with a focused prompt.
2. Give it only the tools it needs.
3. Wrap it in one high-level supervisor tool.
4. Extend `SpecialistName` and the scoper's task instructions.
5. Explain what context should and should not cross the boundary.


In [ ]:
# Activity #1 workspace
# NEW_SPECIALIST_PROMPT = ...
# new_specialist = create_agent(...)
#
# @tool(...)
# async def research_new_specialty(task_json: str) -> str:
#     ...

### YOUR CODE HERE


### 🤔 Activity #1 Reflections

- What unique responsibility did the specialist receive?
- Why is a separate context window useful for this responsibility?
- What additional latency and cost does the specialist introduce?

##### ✅ Answer:

_Write your answer here._


---
# Breakout Room #2
## End-to-End Deep Research Workflow

`create_agent` is excellent for a model-tools loop. It should not be forced to own
every deterministic lifecycle rule.

LangGraph will make these application stages explicit:

```text
scope -> clarify? -> research -> verify -> write -> audit -> evaluate
```


## Task 7: Build the Scoper and Clarification Path

The scoper converts an open-ended request into exactly three bounded research tasks.
If a request is too ambiguous, it returns one high-value clarification question.

LangGraph, not the scoper, owns the pause and resume behavior.


In [ ]:
SCOPER_PROMPT = f'''
You scope veterinary deep-research requests. Today's date is {TODAY}.

Return a ScopeDecision.

Ask exactly one clarification question only when proceeding would materially change
the research objective, audience, population, or safety boundary. Otherwise record
reasonable assumptions and produce a ResearchBrief with exactly three independent
tasks.

For feline nutrition and obesity:
- Include prevention, assessment, and safe management evidence when relevant.
- Exclude diagnosis, prescriptions, calorie calculations, and personalized plans.
- Assign each task to guidelines or evidence.
- Make tasks independent enough to research concurrently.
- Include a veterinary-review safety constraint.
'''

scoper_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SCOPER_PROMPT,
    response_format=ScopeDecision,
    name="research_scoper",
)


## Task 8: Build Verification, Writing, and Evaluation Agents

Verification is separate from writing so the writer cannot quietly promote a weak
source while composing polished prose.

The writer receives only the verification report. The evaluator receives the final
report, verification report, and deterministic citation audit.


In [ ]:
VERIFIER_PROMPT = f'''
You verify a multi-agent veterinary research dossier. Today's date is {TODAY}.

Use Tavily Extract to inspect candidate URLs, preferably in batches. Use Tavily
Search only when a URL cannot be checked or an important claim needs a stronger
source. Approve a source only if the URL resolves through a tool result and the
source supports the associated claim. Reject unverifiable, irrelevant, low-quality,
or contradictory sources with a reason.

Return VerificationReport. Keep only claims supported by approved source URLs.
Preserve material disagreement and uncertainty. Do not invent replacement URLs.
Do not provide individualized veterinary advice.
'''

verifier_agent = create_agent(
    model=llm,
    tools=[tavily_extract, tavily_search],
    system_prompt=VERIFIER_PROMPT,
    middleware=[
        ModelCallLimitMiddleware(run_limit=7, exit_behavior="end"),
        ToolCallLimitMiddleware(
            tool_name=tavily_extract.name,
            run_limit=3,
            exit_behavior="continue",
        ),
        ToolCallLimitMiddleware(
            tool_name=tavily_search.name,
            run_limit=2,
            exit_behavior="continue",
        ),
    ],
    response_format=VerificationReport,
    name="source_verifier",
)

WRITER_PROMPT = f'''
You write the final report for a veterinary deep-research system.
Today's date is {TODAY}.

Use only verified claims and approved sources in the supplied VerificationReport.
Do not introduce external facts or URLs. Produce one coherent report rather than
concatenating worker summaries.

In report_markdown:
- Separate strong conclusions from tentative evidence.
- Explain meaningful disagreement.
- Use numbered citation markers such as [1], [2], and [3].
- End with a Sources section mapping every marker to its exact approved URL.
- Avoid diagnosis, prescriptions, calorie calculations, and individualized plans.
- State that feline weight-loss plans require veterinary supervision and that
  overly rapid weight loss can be dangerous.

The citations field must contain the exact URLs in marker order.
'''

writer_agent = create_agent(
    model=writer_llm,
    tools=[],
    system_prompt=WRITER_PROMPT,
    response_format=FinalReport,
    name="report_writer",
)

EVALUATOR_PROMPT = '''
You evaluate a deep-research report against its brief, verification report, and
deterministic citation audit.

Score each criterion from 1 to 5. Pass only when:
- every score is at least 3,
- citation_integrity and medical_safety are at least 4,
- the deterministic citation audit passed.

Do not reward confident wording when evidence is weak.
'''

evaluator_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=EVALUATOR_PROMPT,
    response_format=ReportEvaluation,
    name="report_evaluator",
)


## Task 9: Define LangGraph State and Nodes

The graph state is the application record. Each node reads the fields it needs and
writes a typed result for the next stage.


In [ ]:
class DeepResearchState(TypedDict, total=False):
    query: str
    clarification_answer: str
    scope_decision: ScopeDecision
    brief: ResearchBrief
    dossier: ResearchDossier
    verification: VerificationReport
    report: FinalReport
    citation_audit: CitationAudit
    evaluation: ReportEvaluation
    errors: list[str]


In [ ]:
async def scope_node(
    state: DeepResearchState,
    config: RunnableConfig,
) -> dict:
    clarification = state.get("clarification_answer")
    user_content = state["query"]
    if clarification:
        user_content += (
            "\n\nThe user answered the clarification question:\n"
            f"{clarification}"
        )

    result = await scoper_agent.ainvoke(
        {"messages": [{"role": "user", "content": user_content}]},
        config=config,
    )
    decision = ScopeDecision.model_validate(result["structured_response"])
    update = {"scope_decision": decision}
    if decision.brief is not None:
        update["brief"] = decision.brief
    return update


def route_after_scope(state: DeepResearchState) -> Literal["clarify", "research"]:
    decision = state["scope_decision"]
    if decision.needs_clarification or decision.brief is None:
        return "clarify"
    return "research"


def clarify_node(state: DeepResearchState) -> dict:
    decision = state["scope_decision"]
    question = (
        decision.clarification_question
        or "What outcome and audience should this research focus on?"
    )
    answer = interrupt(
        {
            "question": question,
            "original_query": state["query"],
        }
    )
    return {"clarification_answer": str(answer)}


In [ ]:
def empty_dossier(brief: ResearchBrief, error: Exception) -> ResearchDossier:
    return ResearchDossier(
        brief_title=brief.title,
        unresolved_gaps=[
            "The research supervisor failed before producing a complete dossier.",
            f"{type(error).__name__}: {error}",
        ],
    )


async def research_node(
    state: DeepResearchState,
    config: RunnableConfig,
) -> dict:
    brief = state["brief"]
    try:
        result = await research_supervisor.ainvoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": (
                            "Coordinate this research brief. Delegate every task "
                            "and return the structured dossier:\n"
                            f"{brief.model_dump_json(indent=2)}"
                        ),
                    }
                ]
            },
            config=config,
        )
        dossier = ResearchDossier.model_validate(
            result["structured_response"]
        )
        dossier = sanitize_dossier(
            dossier,
            brief,
            observed_tool_urls(result),
        )
        return {"dossier": dossier}
    except Exception as exc:
        return {
            "dossier": empty_dossier(brief, exc),
            "errors": state.get("errors", [])
            + [f"Research stage: {type(exc).__name__}: {exc}"],
        }


async def verify_node(
    state: DeepResearchState,
    config: RunnableConfig,
) -> dict:
    dossier = state["dossier"]
    try:
        result = await verifier_agent.ainvoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": dossier.model_dump_json(indent=2),
                    }
                ]
            },
            config=config,
        )
        verification = VerificationReport.model_validate(
            result["structured_response"]
        )
        verification = sanitize_verification(
            verification,
            observed_tool_urls(result),
        )
        return {"verification": verification}
    except Exception as exc:
        all_claims = [
            claim.claim
            for finding in dossier.findings
            for claim in finding.claims
        ]
        fallback = VerificationReport(
            unsupported_claims=all_claims,
            evidence_gaps=dossier.unresolved_gaps
            + ["Verification did not complete."],
            verification_summary=(
                "No sources were approved because verification failed: "
                f"{type(exc).__name__}: {exc}"
            ),
        )
        return {
            "verification": fallback,
            "errors": state.get("errors", [])
            + [f"Verification stage: {type(exc).__name__}: {exc}"],
        }


In [ ]:
async def write_node(
    state: DeepResearchState,
    config: RunnableConfig,
) -> dict:
    verification = state["verification"]
    brief = state["brief"]
    try:
        result = await writer_agent.ainvoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": (
                            "Research brief:\n"
                            f"{brief.model_dump_json(indent=2)}\n\n"
                            "Verified evidence:\n"
                            f"{verification.model_dump_json(indent=2)}"
                        ),
                    }
                ]
            },
            config=config,
        )
        report = FinalReport.model_validate(result["structured_response"])
        return {"report": report}
    except Exception as exc:
        fallback = FinalReport(
            title=brief.title,
            executive_summary=(
                "A reliable report could not be generated from the available "
                "verified evidence."
            ),
            report_markdown=(
                "# Research incomplete\n\n"
                "The writing stage failed. Review the verification record and "
                "rerun the workflow before using any conclusions."
            ),
            limitations=[
                "The report-writing stage did not complete.",
                f"{type(exc).__name__}: {exc}",
            ],
            medical_safety_note=(
                "Consult a veterinarian before changing a cat's diet or "
                "weight-management plan."
            ),
            citations=[],
        )
        return {
            "report": fallback,
            "errors": state.get("errors", [])
            + [f"Writing stage: {type(exc).__name__}: {exc}"],
        }


## Task 10: Deterministic Citation Auditing

LLM verification and LLM evaluation are still model judgments. The citation audit is
ordinary code. It checks whether every report URL belongs to the approved source
ledger, whether URLs are duplicated, and whether numeric markers reference missing
citation entries.


In [ ]:
CITATION_MARKER_PATTERN = re.compile(r"\[(\d+)\]")


def audit_report_citations(
    report: FinalReport,
    verification: VerificationReport,
) -> CitationAudit:
    approved = {
        normalize_url(source.url)
        for source in verification.approved_sources
    }
    ordered_citations = [
        normalize_url(url)
        for url in report.citations
    ]
    urls_in_markdown = urls_in_value(report.report_markdown)
    cited = set(ordered_citations) | urls_in_markdown

    seen: set[str] = set()
    duplicates = []
    for url in ordered_citations:
        if url in seen and url not in duplicates:
            duplicates.append(url)
        seen.add(url)

    marker_numbers = {
        int(value)
        for value in CITATION_MARKER_PATTERN.findall(report.report_markdown)
    }
    missing_markers = sorted(
        number
        for number in marker_numbers
        if number < 1 or number > len(ordered_citations)
    )

    unknown = sorted(cited - approved)
    approved_but_uncited = sorted(approved - cited)
    passed = bool(cited) and not unknown and not duplicates and not missing_markers

    return CitationAudit(
        cited_urls=sorted(cited),
        unknown_urls=unknown,
        duplicate_urls=duplicates,
        approved_but_uncited=approved_but_uncited,
        missing_marker_numbers=missing_markers,
        passed=passed,
    )


def audit_node(state: DeepResearchState) -> dict:
    return {
        "citation_audit": audit_report_citations(
            state["report"],
            state["verification"],
        )
    }


In [ ]:
async def evaluate_node(
    state: DeepResearchState,
    config: RunnableConfig,
) -> dict:
    try:
        payload = {
            "brief": state["brief"].model_dump(),
            "verification": state["verification"].model_dump(),
            "report": state["report"].model_dump(),
            "citation_audit": state["citation_audit"].model_dump(),
        }
        result = await evaluator_agent.ainvoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": json.dumps(payload, indent=2),
                    }
                ]
            },
            config=config,
        )
        evaluation = ReportEvaluation.model_validate(
            result["structured_response"]
        )
        if not state["citation_audit"].passed and evaluation.passed:
            evaluation = evaluation.model_copy(
                update={
                    "passed": False,
                    "improvements": evaluation.improvements
                    + ["Resolve deterministic citation-audit failures."],
                }
            )
        return {"evaluation": evaluation}
    except Exception as exc:
        fallback = ReportEvaluation(
            coverage=1,
            synthesis=1,
            source_quality=1,
            citation_integrity=1,
            uncertainty_handling=1,
            medical_safety=1,
            strengths=[],
            improvements=[
                "The evaluator did not complete.",
                f"{type(exc).__name__}: {exc}",
            ],
            passed=False,
        )
        return {
            "evaluation": fallback,
            "errors": state.get("errors", [])
            + [f"Evaluation stage: {type(exc).__name__}: {exc}"],
        }


## Task 11: Compile and Visualize the Workflow

`InMemorySaver` is required for interrupt/resume because the graph must recover the
state associated with a `thread_id`.


In [ ]:
graph_builder = StateGraph(DeepResearchState)
graph_builder.add_node("scope", scope_node)
graph_builder.add_node("clarify", clarify_node)
graph_builder.add_node("research", research_node)
graph_builder.add_node("verify", verify_node)
graph_builder.add_node("write", write_node)
graph_builder.add_node("audit", audit_node)
graph_builder.add_node("evaluate", evaluate_node)

graph_builder.add_edge(START, "scope")
graph_builder.add_conditional_edges(
    "scope",
    route_after_scope,
    {
        "clarify": "clarify",
        "research": "research",
    },
)
graph_builder.add_edge("clarify", "scope")
graph_builder.add_edge("research", "verify")
graph_builder.add_edge("verify", "write")
graph_builder.add_edge("write", "audit")
graph_builder.add_edge("audit", "evaluate")
graph_builder.add_edge("evaluate", END)

deep_research_graph = graph_builder.compile(
    checkpointer=InMemorySaver()
)

print("Compiled deep_research_graph.")


In [ ]:
try:
    display(Image(deep_research_graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Could not render Mermaid PNG. Mermaid source follows:\n")
    print(deep_research_graph.get_graph().draw_mermaid())
    print(f"\nRender error: {exc}")


#### ❓Question #3

Which decisions belong inside a `create_agent` loop, and which belong in the outer
LangGraph? Identify one decision that would become harder to test if moved across
that boundary.

##### ✅ Answer:

_Write your answer here._


### Test the Citation Auditor with Synthetic Data

These tests do not call a model or search API. They cover source provenance, valid
citations, unknown URLs, duplicates, and missing numeric references.


In [ ]:
synthetic_sources = [
    SourceRecord(
        url="https://example.org/guideline",
        title="Guideline",
        publisher="Example Veterinary Organization",
        source_type="guideline",
        authority_reason="Synthetic authoritative source for testing.",
        relevant_excerpt="Synthetic excerpt.",
        accessed_on=TODAY,
    ),
    SourceRecord(
        url="https://example.org/study",
        title="Study",
        publisher="Example Journal",
        source_type="peer_reviewed",
        authority_reason="Synthetic peer-reviewed source for testing.",
        relevant_excerpt="Synthetic excerpt.",
        accessed_on=TODAY,
    ),
]

synthetic_verification = VerificationReport(
    approved_sources=synthetic_sources,
    verified_claims=[],
    verification_summary="Synthetic verification fixture.",
)

invented_source = synthetic_sources[0].model_copy(
    update={"url": "https://example.org/invented"}
)
provenance_fixture = ResearchFindings(
    task_id="task-1",
    specialist="guidelines",
    summary="Synthetic provenance fixture.",
    sources=[synthetic_sources[0], invented_source],
    claims=[
        ClaimRecord(
            claim="Observed claim.",
            source_urls=[synthetic_sources[0].url],
            confidence="high",
            caveat="Synthetic.",
        ),
        ClaimRecord(
            claim="Invented claim.",
            source_urls=[invented_source.url],
            confidence="low",
            caveat="Synthetic.",
        ),
    ],
)
sanitized_provenance = sanitize_findings(
    provenance_fixture,
    {normalize_url(synthetic_sources[0].url)},
)
assert [source.url for source in sanitized_provenance.sources] == [
    synthetic_sources[0].url
]
assert [claim.claim for claim in sanitized_provenance.claims] == [
    "Observed claim."
]

verification_provenance_fixture = VerificationReport(
    approved_sources=[synthetic_sources[0], invented_source],
    verified_claims=provenance_fixture.claims,
    verification_summary="Synthetic verifier provenance fixture.",
)
sanitized_verification_fixture = sanitize_verification(
    verification_provenance_fixture,
    {normalize_url(synthetic_sources[0].url)},
)
assert [source.url for source in sanitized_verification_fixture.approved_sources] == [
    synthetic_sources[0].url
]
assert [claim.claim for claim in sanitized_verification_fixture.verified_claims] == [
    "Observed claim."
]

valid_report = FinalReport(
    title="Valid",
    executive_summary="Synthetic.",
    report_markdown=(
        "A supported statement [1]. Another supported statement [2].\n\n"
        "Sources:\n"
        "1. https://example.org/guideline\n"
        "2. https://example.org/study"
    ),
    limitations=[],
    medical_safety_note="Synthetic.",
    citations=[
        "https://example.org/guideline",
        "https://example.org/study",
    ],
)

invalid_report = valid_report.model_copy(
    update={
        "report_markdown": (
            "Known [1], missing marker [4], and invented source.\n"
            "https://example.org/invented"
        ),
        "citations": [
            "https://example.org/guideline",
            "https://example.org/guideline",
        ],
    }
)

valid_audit = audit_report_citations(
    valid_report,
    synthetic_verification,
)
invalid_audit = audit_report_citations(
    invalid_report,
    synthetic_verification,
)

assert valid_audit.passed
assert "https://example.org/invented" in invalid_audit.unknown_urls
assert "https://example.org/guideline" in invalid_audit.duplicate_urls
assert 4 in invalid_audit.missing_marker_numbers

print("Synthetic provenance and citation-audit tests passed.")
print(invalid_audit.model_dump_json(indent=2))


## Task 12: Stream and Run the Full Deep Research System

`run_deep_research` streams graph updates with nested namespaces. Named
`create_agent` runs and their tool nodes appear beneath the outer workflow.

When scoping interrupts, the function returns a paused status. Resume the same
`thread_id` with `resume_deep_research`.


In [ ]:
def print_stream_part(part: dict) -> None:
    namespace = part.get("ns", ())
    data = part.get("data", part)
    label = " -> ".join(namespace) if namespace else "outer-graph"

    if not isinstance(data, dict):
        return

    for node_name, update in data.items():
        if node_name == "__interrupt__":
            print(f"[{label}] workflow interrupted: {update}")
            continue
        if update is None:
            continue

        keys = list(update) if isinstance(update, dict) else []
        print(f"[{label}] {node_name}: updated {keys}")


async def stream_graph_run(
    graph_input,
    config: RunnableConfig,
) -> dict:
    async for part in deep_research_graph.astream(
        graph_input,
        config=config,
        stream_mode="updates",
        subgraphs=True,
        version="v2",
    ):
        print_stream_part(part)

    snapshot = await deep_research_graph.aget_state(config)
    values = dict(snapshot.values)
    if snapshot.next:
        question = None
        decision = values.get("scope_decision")
        if decision is not None:
            question = decision.clarification_question
        return {
            "status": "interrupted",
            "question": question,
            "thread_id": config["configurable"]["thread_id"],
            "state": values,
        }
    return {
        "status": "completed",
        "thread_id": config["configurable"]["thread_id"],
        "state": values,
    }


async def run_deep_research(
    query: str,
    thread_id: str,
) -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    return await stream_graph_run(
        {"query": query, "errors": []},
        config,
    )


async def resume_deep_research(
    answer: str,
    thread_id: str,
) -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    return await stream_graph_run(
        Command(resume=answer),
        config,
    )


### Optional Clarification Demonstration

A broad request may pause:

```python
paused = await run_deep_research(
    "Research cat food.",
    thread_id="ambiguous-cat-food",
)
paused["question"]
```

Resume the exact saved workflow:

```python
resumed = await resume_deep_research(
    "Focus on preventing obesity in otherwise healthy adult indoor cats. "
    "The audience is cat owners.",
    thread_id="ambiguous-cat-food",
)
```


### Full Demonstration

This run can consume multiple model and Tavily calls. Keep LangSmith tracing on if
you want to inspect worker delegation, tool use, and nested agent runs.


In [ ]:
research_query = (
    "What does current evidence say about preventing and safely managing "
    "obesity in adult indoor cats?"
)

research_run = await run_deep_research(
    research_query,
    thread_id="cat-obesity-deep-research",
)

print(research_run["status"])
if research_run["status"] == "interrupted":
    print("Clarification requested:", research_run["question"])


If the run paused, answer the clarification question with
`resume_deep_research(...)`. When complete, display the final report and quality
records.


In [ ]:
if research_run["status"] == "completed":
    final_state = research_run["state"]
    display(Markdown(final_state["report"].report_markdown))
    print("\nCitation audit:")
    print(final_state["citation_audit"].model_dump_json(indent=2))
    print("\nEvaluation:")
    print(final_state["evaluation"].model_dump_json(indent=2))
    if final_state.get("errors"):
        print("\nRecorded partial failures:")
        print("\n".join(f"- {item}" for item in final_state["errors"]))


#### ❓Question #4

Inspect the trace or streamed updates:

- Did the supervisor call independent workers in one turn?
- Which worker used more searches or model calls?
- Which claims were rejected or weakened during verification?
- Did the final writer preserve disagreements?

##### ✅ Answer:

_Write your answer here._


## 🏗️ Activity #2: Compare Research Depth and Cost

Run the same query twice with two budget profiles.

**Profile A: Quick**

```bash
AIM_SEARCH_DEPTH=basic
AIM_SEARCH_CALL_LIMIT=1
AIM_EXTRACT_CALL_LIMIT=1
AIM_WORKER_MODEL_CALL_LIMIT=4
```

**Profile B: Deep**

```bash
AIM_SEARCH_DEPTH=advanced
AIM_SEARCH_CALL_LIMIT=4
AIM_EXTRACT_CALL_LIMIT=3
AIM_WORKER_MODEL_CALL_LIMIT=10
```

Restart the kernel after changing environment variables.

Compare:

- Number and quality of approved sources
- Coverage and disagreement discovered
- Citation-audit result
- Evaluator scores
- LangSmith latency, token use, and tool calls


In [ ]:
# Activity #2 results
budget_comparison = {
    "quick": {
        "approved_sources": None,
        "latency_seconds": None,
        "tool_calls": None,
        "evaluation_passed": None,
        "notes": "",
    },
    "deep": {
        "approved_sources": None,
        "latency_seconds": None,
        "tool_calls": None,
        "evaluation_passed": None,
        "notes": "",
    },
}

budget_comparison


### 🤔 Activity #2 Reflections

- Which additional research improved the report?
- Where did deeper research add cost without changing the conclusion?
- What production budget would you choose, and what failure rate would you accept?

##### ✅ Answer:

_Write your answer here._


## Advanced Build: Add a Local-Corpus Specialist

Add the Session 2 or Session 3 cat-health corpus as a third research vertical.

Requirements:

1. Build a retriever tool over the local corpus.
2. Create a named local-corpus `create_agent` with only that tool.
3. Wrap it as a supervisor tool.
4. Label local sources distinctly from current web sources.
5. Ask the verifier to identify outdated or conflicting local guidance.
6. Keep the final writer limited to verified evidence.

This extension demonstrates a common enterprise architecture:

```text
public web evidence + organization knowledge + one verification layer
```


## Final Takeaways

1. Multi-agent systems are primarily a context-engineering choice.
2. Specialized workers should return compact results, not their full histories.
3. `create_agent` owns dynamic reasoning and tool use inside an actor.
4. LangGraph owns deterministic lifecycle, branching, persistence, and resume.
5. Verification should be separate from final writing.
6. URLs need a deterministic ledger and audit; model confidence is not enough.
7. Research depth is a tunable cost, latency, and quality policy.
8. Partial failures should become visible gaps, never fabricated evidence.

Ask your instructor or coding agent about any trace, tool call, schema, or graph edge
that is unclear.
